# Multi-Level Driver Drowsiness Detection Using Hybrid Facial Features and Machine Learning

**Author**: Archee Jaiswal  
**Category**: Computer Vision / Machine Learning Application  

### System Overview:
- **Data & Ingestion**: UTA-RLDD (University of Texas at Arlington Real-Life Drowsiness Dataset) 3-class classification (Alert: 0, Low Vigilant: 1, Drowsy: 2)  
- **Vision & Features**: MediaPipe 468/478 landmarks, Eye Aspect Ratio (EAR), Mouth Aspect Ratio (MAR), solvePnP 3D Head Pose, Eyebrow dynamics, and adaptive baseline calibration  
- **Ensemble Modeling**: Soft-Voting Ensemble blending Random Forest, XGBoost, and Multilayer Perceptron (MLP)  
- **Validation & Ablation**: Seen-Subject Stratified CV, Unseen-Subject GroupKFold (zero participant leakage), and full feature ablation  

---


## 1. Setup and Imports
Import standard libraries and project modules.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.utils.config import ALL_FEATURE_NAMES, CLASSES, FEATURE_GROUPS
from src.models.train import train_and_save_pipeline
from src.evaluation.validation import ValidationEvaluator
from src.evaluation.ablation import FeatureAblationStudy

print("Loaded configuration and project modules successfully!")

## 2. Dataset Ingestion and Exploration
Loading the multi-level drowsiness dataset (Alert: 0, Low Vigilant: 1, Drowsy: 2).


In [ ]:
data_path = PROJECT_ROOT / "data" / "features" / "utarldd_benchmark_features.csv"
df = pd.read_csv(data_path)

print(f"Total Samples: {len(df)}")
print(f"Unique Subjects: {df['subject_id'].nunique()}")
print("Class Distribution:")
print(df['label'].value_counts().rename(index=CLASSES))

df.head()

## 3. Physiological Feature Distributions
Visualize how Eye Aspect Ratio (EAR), Mouth Aspect Ratio (MAR), Head Pitch, and PERCLOS change across the 3 drowsiness stages.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df_plot = df.copy()
df_plot['Drowsiness State'] = df_plot['label'].map(CLASSES)
order = ["Alert", "Low Vigilant", "Drowsy"]
palette = ["#10B981", "#F59E0B", "#EF4444"]

sns.boxplot(x="Drowsiness State", y="ear_mean", data=df_plot, order=order, palette=palette, ax=axes[0, 0])
axes[0, 0].set_title("Eye Aspect Ratio (EAR Mean)", fontsize=12, fontweight="bold")

sns.boxplot(x="Drowsiness State", y="perclos", data=df_plot, order=order, palette=palette, ax=axes[0, 1])
axes[0, 1].set_title("PERCLOS (Proportion Eye Closure)", fontsize=12, fontweight="bold")

sns.boxplot(x="Drowsiness State", y="mar_max", data=df_plot, order=order, palette=palette, ax=axes[1, 0])
axes[1, 0].set_title("Mouth Aspect Ratio (MAR Max - Yawning)", fontsize=12, fontweight="bold")

sns.boxplot(x="Drowsiness State", y="pitch_std", data=df_plot, order=order, palette=palette, ax=axes[1, 1])
axes[1, 1].set_title("Head Posture Instability (Pitch Std)", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()

## 4. Model Development & Soft-Voting Ensemble Training
Training individual classifiers (Random Forest, XGBoost, MLP) and constructing the Soft-Voting Ensemble.


In [ ]:
ensemble, test_scores = train_and_save_pipeline(df=df, tune_hyperparameters=False, test_size=0.20)

scores_df = pd.DataFrame(list(test_scores.items()), columns=["Model", "Test Accuracy"])
scores_df["Test Accuracy (%)"] = scores_df["Test Accuracy"].apply(lambda x: f"{x*100:.2f}%")
scores_df

## 5. Multi-Protocol Validation & Ablation Study
Evaluation across both Seen-Subject and Unseen-Subject (GroupKFold) validation protocols.


In [ ]:
evaluator = ValidationEvaluator(random_state=42)

# 1. Seen-Subject Cross-Validation
seen_cv = evaluator.run_seen_subject_cv(df=df, n_splits=5)
print("=== Seen-Subject Validation Results ===")
display(seen_cv.groupby('model')[['accuracy', 'f1_macro', 'precision_macro', 'recall_macro']].mean())

# 2. Unseen-Subject Cross-Validation (Generalization)
unseen_seg_cv, unseen_vid_cv = evaluator.run_unseen_subject_cv(df=df, n_splits=5)
print("\n=== Unseen-Subject (Generalization) Validation Results ===")
display(unseen_seg_cv.groupby('model')[['accuracy', 'f1_macro', 'precision_macro', 'recall_macro']].mean())

### Feature Group Ablation Study
Comparing performance across feature subsets (Eye only, Mouth only, Head Pose only, Eyebrow only, and Full Hybrid).

In [ ]:
ablation = FeatureAblationStudy(random_state=42)
ablation_results = ablation.run_study(df=df, n_splits=5, save_plot=True)

# Display results table
display(ablation_results)

# Display plot
from IPython.display import Image
Image(filename=str(PROJECT_ROOT / 'results' / 'ablation_study.png'))